# 04 — Compare & cross-modal retrieval

SONAR puts speech and text in a shared 1024-d space, so the same cosine metric answers:

- speech ↔ speech (same-speaker / same-content lookup)
- text ↔ text (translation / paraphrase ranking)
- speech ↔ text (open-vocabulary retrieval)

This notebook walks through every operand kind the `compare` module accepts, then uses `top_k` / `threshold` to turn a full score matrix into a ranked list.

The grammar at a glance (same for the CLI and the `compare()` library call):

| Operand | Resolves to |
|---|---|
| `path/to/store` | all embeddings in a store → `(N, 1024)` |
| `store:path#segment_id` | one embedding → `(1, 1024)` |
| `file_id,channel,spkid,start,end[,lang]` | inline segment decoded + encoded on the fly |
| `text:<text>@<lang>` | literal text, encoded on the fly |
| `text-file:<path>@<lang>` | one text per line, encoded as a batch |

In [ ]:
import tempfile, wave, hashlib
from pathlib import Path
import numpy as np

from sonar_probe import ExtractConfig, CompareConfig, extract, extract_text, compare, load_store, read_text_tsv
from sonar_probe.sources import get_source
from sonar_probe.sinks import get_sink
from sonar_probe.engine.speech import SpeechEncoderPool
from sonar_probe.engine.text import TextEncoder

WORK = Path(tempfile.mkdtemp(prefix="sonar-probe-compare-"))
print("workdir:", WORK)

DIM = 1024
def _hv(k):
    rng = np.random.default_rng(int.from_bytes(hashlib.sha256(k).digest()[:8], "big"))
    v = rng.standard_normal(DIM).astype(np.float32); return v / (np.linalg.norm(v) + 1e-9)

class FakeSpeechEncoder:
    def encode(self, waves): return np.stack([_hv(w.tobytes()) for w in waves]).astype(np.float32)
class FakeTextBackend:
    def encode(self, texts, lang): return np.stack([_hv(f"{lang}|{t}".encode()) for t in texts]).astype(np.float32)

fake_pool = SpeechEncoderPool(device="cpu", encoder_factory=lambda l, d: FakeSpeechEncoder())
fake_text = TextEncoder(device="cpu", encoder_factory=lambda d: FakeTextBackend())

## Build a tiny speech store and a tiny text store

In [ ]:
SR = 16000
def tone(path, hz, seconds):
    t = np.arange(int(SR * seconds)) / SR
    x = (0.3 * np.sin(2 * np.pi * hz * t) * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(x.tobytes())

audio = WORK / "audio"; audio.mkdir()
tone(audio / "rec01.wav", 440.0, 6.0)
tone(audio / "rec02.wav", 880.0, 6.0)

(WORK / "wav.scp").write_text(f"rec01 {audio/'rec01.wav'}\nrec02 {audio/'rec02.wav'}\n")
(WORK / "segments.tsv").write_text(
    "file_id\tchannel\tspkid\tstart\tend\tlang\n"
    "rec01\t0\tspk1\t0.0\t2.0\teng\n"
    "rec01\t0\tspk1\t2.0\t4.0\teng\n"
    "rec02\t0\tspk2\t0.0\t3.0\teng\n"
    "rec02\t0\tspk2\t3.0\t6.0\teng\n"
)
(WORK / "texts.tsv").write_text(
    "text_id\ttext\tlang\n"
    "q_fox\ta quick brown fox\teng\n"
    "q_dog\tthe lazy dog sleeps\teng\n"
    "q_440\ta tone at four hundred forty hertz\teng\n"
)

src = get_source("plain_text", input=WORK/"segments.tsv", wav_scp=WORK/"wav.scp")
snk = get_sink("kaldi_ark", output=WORK/"speech_store")
extract(src, snk, ExtractConfig(device="cpu", batch_size=4), pool=fake_pool)

snk_t = get_sink("hf_dataset", output=WORK/"text_store")
extract_text(read_text_tsv(WORK/"texts.tsv"), snk_t, ExtractConfig(device="cpu", batch_size=4), encoder=fake_text)

speech, text = load_store(WORK/"speech_store"), load_store(WORK/"text_store")
print("speech ids:", speech.ids())
print("text ids:  ", text.ids())

## 1. Store-id ↔ store-id (scalar)

`compare()` returns a `CompareResult(a_ids, b_ids, modality_a, modality_b, scores)`. When both sides are 1-element, `scores.shape == (1, 1)`.

In [ ]:
a, b = speech.ids()[0], speech.ids()[1]
cfg = CompareConfig(device="cpu")
res = compare(f"store:{WORK/'speech_store'}#{a}", f"store:{WORK/'speech_store'}#{b}",
              cfg, pool=fake_pool, text_encoder=fake_text)
print(f"{res.modality_a} ↔ {res.modality_b}  shape={res.scores.shape}  cosine={float(res.scores[0,0]):+.4f}")

## 2. Text ↔ store (cross-modal)

`text:<text>@<lang>` is encoded on the fly through the text encoder; the right side is a store path (bare), so the result is a `(1, N)` row of cosines against every speech segment.

In [ ]:
res = compare("text:a tone at four hundred forty hertz@eng", str(WORK/"speech_store"),
              cfg, pool=fake_pool, text_encoder=fake_text)
print(f"shape={res.scores.shape}  modality={res.modality_a} ↔ {res.modality_b}")
for b_id, s in zip(res.b_ids, res.scores[0]):
    print(f"  {b_id:30s}  {float(s):+.4f}")

## 3. Inline segment ↔ store

`file_id,channel,spkid,start,end[,lang]` is decoded and encoded live. Requires `CompareConfig.wav_scp=` so the inline operand can resolve `file_id` to audio.

In [ ]:
cfg_inline = CompareConfig(device="cpu", wav_scp=WORK/"wav.scp", speech_lang="eng")
res = compare("rec01,0,spk1,0.0,2.0", str(WORK/"speech_store"),
              cfg_inline, pool=fake_pool, text_encoder=fake_text)
print("inline id →", res.a_ids[0])
print(f"shape={res.scores.shape}")

## 4. Text-file ↔ store (batched queries)

`text-file:<path>@<lang>` reads one line per text and encodes the whole batch at once. Great for query lists.

In [ ]:
qfile = WORK / "queries.txt"
qfile.write_text(
    "a quick brown fox\n"
    "a tone at four hundred forty hertz\n"
    "an eight-eighty hertz tone\n"
)
res = compare(f"text-file:{qfile}@eng", str(WORK/"speech_store"),
              cfg, pool=fake_pool, text_encoder=fake_text)
print(f"shape={res.scores.shape}")
for a_id, row in zip(res.a_ids, res.scores):
    best = int(np.argmax(row))
    print(f"  query {a_id:25s} best→ {res.b_ids[best]} ({float(row[best]):+.4f})")

## 5. `top_k` + `threshold` via `iter_pairs`

`CompareResult.iter_pairs(top_k=None, threshold=None)` yields `(a_id, b_id, score)` tuples. For each row, it sorts by score descending, keeps the top-K, then drops anything below `threshold`.

In [ ]:
res = compare(str(WORK/"text_store"), str(WORK/"speech_store"),
              cfg, pool=fake_pool, text_encoder=fake_text)
print(f"full matrix: {res.scores.shape}\n")
print("top-2 per query, no threshold:")
for a, b, s in res.iter_pairs(top_k=2):
    print(f"  {a:8s} → {b:30s}  {s:+.4f}")

## 6. Full matrix as NumPy

`CompareResult.scores` is a plain `(N, M) float32` ndarray — hand it to pandas / scikit-learn / your plotting library of choice.

In [ ]:
import pandas as pd
df = pd.DataFrame(res.scores, index=res.a_ids, columns=res.b_ids)
df.round(4)

## From the CLI

The same operands work verbatim on the command line:

```bash
sonar-probe compare \
    --a text-file:queries.txt@eng \
    --b path/to/speech_store \
    --top-k 2 \
    --output-format long \
    --output top2.tsv
```